# Segment-level comparison: PyTorch MLP versus legacy sklearn MLP

Both models are evaluated on the same complete physical observation
segments that were entirely absent from the sampled training subset.
The saved `subset_indices.npy` mapping is essential: split indices refer
to subset rows, not directly to rows of the full observation.

This notebook is an external comparison, not a replacement for the
held-out subset test metrics saved by either training notebook.


In [ ]:
from __future__ import annotations

import json
import platform
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd


WORK_ROOT = Path.cwd().resolve()

DATA_FOLDER = Path("/hercules/results/akazantsev/rfim_dataset")
META_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_channels_meta.csv"
SPLIT_PATH = DATA_FOLDER / "split_indices.npz"
PROFILES_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_channels.npy"

# The training subset deliberately retains only statistical features and labels.
# These full files retain channel and segment identity and are used only by the
# inference-timing notebook, where one input must correspond to a real 256-channel
# observation segment.
FULL_META_PATH = DATA_FOLDER / "B0531+21_59000_48386_channels_meta.csv"
FULL_PROFILES_PATH = DATA_FOLDER / "B0531+21_59000_48386_channels.npy"
SUBSET_SOURCE_INDICES_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_indices.npy"

# Change the tag only for a deliberate new experiment. Existing results are never overwritten.
RUN_TAG = "b0531_legacy_performance_v1"
RUN_ROOT = WORK_ROOT / "outputs" / "performance_comparison" / RUN_TAG


def json_ready(value):
    if isinstance(value, dict):
        return {key: json_ready(item) for key, item in value.items()}
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    return value


def write_json(path: Path, payload: dict) -> None:
    with path.open("w", encoding="utf-8") as handle:
        json.dump(json_ready(payload), handle, indent=2, sort_keys=True)
        handle.write("\n")


def git_revision() -> str:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=WORK_ROOT,
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None


In [ ]:
import joblib
import torch
from torch import nn
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SELECTED_FEATURES = ["mean_o", "std_o", "skew_o"]
NSAMP = 256
DEVICE = torch.device("cpu")
pytorch_dir = RUN_ROOT / "mlp_pytorch_orig_top3"
pytorch_checkpoint_path = pytorch_dir / "best_checkpoint.pt"

def find_legacy_bundle(start: Path) -> Path:
    for candidate_root in [start, *start.parents]:
        candidate = candidate_root / "saved_models_selected_topk" / "rfi_model_MLP_orig_top3.joblib"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not find saved_models_selected_topk/rfi_model_MLP_orig_top3.joblib "
        "above the current working directory."
    )

legacy_bundle_path = find_legacy_bundle(WORK_ROOT)
required = [
    pytorch_checkpoint_path,
    legacy_bundle_path,
    FULL_META_PATH,
    SUBSET_SOURCE_INDICES_PATH,
    SPLIT_PATH,
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing comparison inputs:\n" + "\n".join(map(str, missing)))


class MLPOrigTop3Logits(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1)
        )

    def forward(self, features):
        return self.layers(features).squeeze(-1)


In [ ]:
full_meta = pd.read_csv(FULL_META_PATH)
full_meta["label"] = full_meta["label"].fillna("None")
required_columns = {"segment_index", "channel_index", "label", *SELECTED_FEATURES}
missing_columns = required_columns.difference(full_meta.columns)
if missing_columns:
    raise ValueError(f"Full metadata is missing required columns: {sorted(missing_columns)}")

subset_source_indices = np.asarray(np.load(SUBSET_SOURCE_INDICES_PATH), dtype=int)
splits = np.load(SPLIT_PATH)
used_subset_rows = np.concatenate([
    np.asarray(splits["train_idx"], dtype=int),
    np.asarray(splits["val_idx"], dtype=int),
    np.asarray(splits["test_idx"], dtype=int),
])
if len(subset_source_indices) <= used_subset_rows.max():
    raise ValueError("The subset-index mapping is shorter than the split indices.")
if subset_source_indices.max() >= len(full_meta):
    raise ValueError("The subset-index mapping does not refer to rows of the supplied full metadata.")

used_source_rows = set(subset_source_indices[used_subset_rows].tolist())
complete_unseen_segments = []
for segment_index, group in full_meta.groupby("segment_index", sort=True):
    if len(group) != NSAMP or group["channel_index"].nunique() != NSAMP:
        continue
    source_rows = group.index.to_numpy(dtype=int)
    if not any(row in used_source_rows for row in source_rows):
        complete_unseen_segments.append((int(segment_index), source_rows))

if not complete_unseen_segments:
    raise ValueError(
        "No complete segment remains entirely outside the sampled subset. "
        "Do not compare models on partly used segments without stating that limitation."
    )

comparison_rows = np.concatenate([rows for _, rows in complete_unseen_segments])
comparison_meta = full_meta.iloc[comparison_rows].copy()
comparison_meta = comparison_meta.sort_values(["segment_index", "channel_index"])
x_comparison = (
    comparison_meta[SELECTED_FEATURES]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
    .to_numpy(dtype=np.float32)
)
y_comparison = comparison_meta["label"].eq("NBRFI").to_numpy(dtype=int)

print(f"Complete unseen physical segments: {len(complete_unseen_segments)}")
print(f"Rows in comparison: {len(comparison_meta)}")


In [ ]:
legacy_bundle = joblib.load(legacy_bundle_path)
legacy_pipeline = legacy_bundle["pipeline"]
legacy_features = list(legacy_bundle["feature_cols"])
legacy_threshold = float(legacy_bundle["threshold"])
if legacy_features != SELECTED_FEATURES:
    raise ValueError(f"Unexpected legacy feature order: {legacy_features}")

checkpoint = torch.load(pytorch_checkpoint_path, map_location=DEVICE)
if checkpoint["feature_cols"] != SELECTED_FEATURES:
    raise ValueError("PyTorch checkpoint feature order does not match the legacy model.")
pytorch_model = MLPOrigTop3Logits().to(DEVICE)
pytorch_model.load_state_dict(checkpoint["model_state_dict"])
pytorch_model.eval()

with torch.no_grad():
    pytorch_scores = torch.sigmoid(
        pytorch_model(torch.from_numpy(x_comparison).to(DEVICE))
    ).cpu().numpy()
legacy_scores = legacy_pipeline.predict_proba(x_comparison)[:, 1]
pytorch_threshold = float(json.load((pytorch_dir / "training_summary.json").open())["threshold"])

predictions = {
    "legacy_sklearn": (legacy_scores, legacy_threshold),
    "pytorch": (pytorch_scores, pytorch_threshold),
}


In [ ]:
def metrics_for_rows(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    precision, recall, fscore, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=1, zero_division=0
    )
    return {
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision), "recall": float(recall), "f1": float(fscore),
    }


per_segment_rows, overall_rows = [], []
segment_values = comparison_meta["segment_index"].to_numpy()
for model_name, (scores, threshold) in predictions.items():
    overall_rows.append({
        "model": model_name,
        "threshold": threshold,
        "n_rows": len(y_comparison),
        "n_segments": len(complete_unseen_segments),
        **metrics_for_rows(y_comparison, scores, threshold),
    })

    for segment_index in tqdm(np.unique(segment_values), desc=f"Segment metrics: {model_name}"):
        mask = segment_values == segment_index
        per_segment_rows.append({
            "model": model_name,
            "segment_index": int(segment_index),
            "threshold": threshold,
            "n_channels": int(mask.sum()),
            "n_positive_channels": int(y_comparison[mask].sum()),
            **metrics_for_rows(y_comparison[mask], scores[mask], threshold),
        })

output_dir = RUN_ROOT / "mlp_pytorch_vs_legacy_segment_comparison"
if output_dir.exists():
    raise FileExistsError(f"{output_dir} already exists. Choose a new RUN_TAG rather than overwrite it.")
output_dir.mkdir(parents=True)

per_segment = pd.DataFrame(per_segment_rows)
overall = pd.DataFrame(overall_rows)
per_segment.to_csv(output_dir / "per_segment_metrics.csv", index=False)
overall.to_csv(output_dir / "overall_metrics.csv", index=False)
display(overall)


In [ ]:
wide = per_segment.pivot(index="segment_index", columns="model", values=["accuracy", "f1"])
fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.2))
for ax, metric in zip(axes, ["accuracy", "f1"]):
    x = wide[(metric, "legacy_sklearn")]
    y = wide[(metric, "pytorch")]
    ax.scatter(x, y, s=18, alpha=0.7)
    limits = [min(x.min(), y.min()), max(x.max(), y.max())]
    ax.plot(limits, limits, color="black", linestyle="--", linewidth=1)
    ax.set(
        xlabel=f"Legacy sklearn {metric}",
        ylabel=f"PyTorch {metric}",
        title=f"Per-segment {metric}",
        xlim=limits,
        ylim=limits,
    )
    ax.grid(linestyle="--", alpha=0.35)
fig.tight_layout()
fig.savefig(output_dir / "per_segment_metric_comparison.png", dpi=300, bbox_inches="tight")
fig.savefig(output_dir / "per_segment_metric_comparison.pdf", bbox_inches="tight")
plt.show()

protocol = {
    "comparison_scope": "complete physical segments whose 256 source rows are absent from train, validation, and test subset splits",
    "full_metadata_path": FULL_META_PATH,
    "subset_source_indices_path": SUBSET_SOURCE_INDICES_PATH,
    "legacy_bundle_path": legacy_bundle_path,
    "pytorch_checkpoint_path": pytorch_checkpoint_path,
    "features": SELECTED_FEATURES,
    "n_segments": len(complete_unseen_segments),
    "n_rows": len(comparison_meta),
    "code_revision": git_revision(),
}
write_json(output_dir / "comparison_protocol.json", protocol)
print(f"Saved comparison tables and figure in {output_dir}")
